# 09 - 每日增量管线

本 Notebook 完成以下任务：
1. 管线设计说明（早盘/午盘双时段）
2. 手动运行管线
3. cron 定时任务配置
4. 查看运行日志

---

## 管线设计

### 双时段调度

| 时段 | 触发时间 | 执行内容 | 目的 |
|------|---------|---------|------|
| **morning** | 11:35 | 日历 + 日线 + ETF + 指数 | 早盘行情快照 |
| **afternoon** | 15:05 | 全部 10 步 | 完整日终数据 + 校验 |

### morning 步骤（早盘后）
```
1. 交易日历同步        trade_calendar
2. 个股日线增量          stock_daily
3. ETF 日线增量          etf_daily
4. 指数日线增量          index_daily
```

### afternoon 步骤（午盘后）
```
1. 交易日历同步          trade_calendar
2. 股票/ETF 列表同步      stock_info, etf_info
3. 个股日线增量            stock_daily          ← 覆盖早盘半日数据
4. ETF 日线增量            etf_daily
5. 指数日线增量            index_daily
6. 每日估值增量            stock_fundamental
7. 融资融券增量            stock_margin
8. 股东增减持              stock_holder_trade
9. 股东户数                stock_holder_count
10. 数据质量校验           (不写表)
```

### 增量机制
所有行情类数据通过 `trade_calendar` 比对缺失交易日，只拉缺失部分。
午盘管线会用 Upsert 覆盖早盘的半日数据，保证最终是完整日线。

### 容错
每个步骤独立 try-except，单步失败不影响后续步骤。

## 1. 手动运行管线

按需选择运行以下某一个 cell：
- **早盘** (morning)：11:30 收盘后运行，拉取日线快照（半日数据）
- **午盘** (afternoon)：15:00 收盘后运行，拉取完整日数据 + 财务/事件
- **全量** (full)：包含以上全部步骤

> 注意：运行后数据入库，`13_daily_advisor.ipynb` 就能基于最新数据生成信号

In [6]:
from invest_model.pipeline.daily_pipeline import DailyPipeline
from datetime import datetime

pipeline = DailyPipeline()

def _run_and_print(session: str):
    """运行管线并打印结果"""
    print(f'{"=" * 60}')
    print(f'  运行时间: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    print(f'  模式: {session}')
    print(f'{"=" * 60}\n')
    results = pipeline.run(session=session)
    print(f'\n{"=" * 60}')
    print(f'运行结果汇总:')
    print(f'{"=" * 60}')
    for step, status in results.items():
        icon = '✅' if 'OK' in status else '❌'
        print(f'  {icon} {step}: {status}')
    return results

14:35:36 | INFO    | Tushare 使用自定义接口基地址: http://118.89.66.41:8010/
14:35:36 | INFO    | Tushare 客户端初始化完成（Token 验证通过）


### 1.1 早盘运行 (11:30 收盘后)

拉取当天上午盘日线快照，运行后即可在 `13_daily_advisor` 中看到基于上午数据的信号。

In [7]:
# 早盘 (上午11:30收盘后运行)
# 步骤: 交易日历 → 个股日线 → ETF日线 → 指数日线
results_morning = _run_and_print("morning")

14:35:36 | INFO    | ============================================================
14:35:36 | INFO    | 每日管线启动 [morning]: 2026-05-06 14:35:36
14:35:36 | INFO    | 执行步骤: ['calendar', 'stock_daily', 'etf_daily', 'index_daily']
14:35:36 | INFO    | ============================================================
14:35:36 | INFO    | --- [交易日历] 开始 ---
14:35:36 | INFO    | 交易日历已覆盖到 20261231，无需更新
14:35:36 | INFO    | --- [交易日历] 完成: 0 ---
14:35:36 | INFO    | --- [个股日线] 开始 ---


  运行时间: 2026-05-06 14:35:36
  模式: morning



14:35:39 | INFO    | 日线增量完成: 10 只, 新增 0 条
14:35:39 | INFO    | --- [个股日线] 完成: 10 只, 新增 0 条 ---
14:35:39 | INFO    | --- [ETF日线] 开始 ---
14:35:39 | INFO    | --- [ETF日线] 完成: 1 只, 新增 0 条 ---
14:35:39 | INFO    | --- [指数日线] 开始 ---
14:35:41 | INFO    | --- [指数日线] 完成: 5 指数, 新增 0 条 ---
14:35:41 | INFO    | ============================================================
14:35:41 | INFO    | 每日管线完成 [morning]，耗时 4.4s
14:35:41 | INFO    |   calendar: OK (0)
14:35:41 | INFO    |   stock_daily: OK (10 只, 新增 0 条)
14:35:41 | INFO    |   etf_daily: OK (1 只, 新增 0 条)
14:35:41 | INFO    |   index_daily: OK (5 指数, 新增 0 条)
14:35:41 | INFO    | ============================================================



运行结果汇总:
  ✅ calendar: OK (0)
  ✅ stock_daily: OK (10 只, 新增 0 条)
  ✅ etf_daily: OK (1 只, 新增 0 条)
  ✅ index_daily: OK (5 指数, 新增 0 条)


### 1.2 午盘运行 (15:00 收盘后)

拉取完整日数据 + 财务/融资融券/北向资金等，并生成综合评分和操作建议。

In [8]:
# 午盘 (下午15:00收盘后运行)
# 步骤: 全部数据拉取 + 技术指标 + 综合评分 + 操作建议 + 数据校验
results_afternoon = _run_and_print("afternoon")

14:35:41 | INFO    | ============================================================
14:35:41 | INFO    | 每日管线启动 [afternoon]: 2026-05-06 14:35:41
14:35:41 | INFO    | 执行步骤: ['calendar', 'stock_list', 'stock_daily', 'etf_daily', 'index_daily', 'daily_basic', 'technical', 'margin', 'cashflow', 'northbound', 'holder_trade', 'holder_count', 'signal_generation', 'validation']
14:35:41 | INFO    | ============================================================
14:35:41 | INFO    | --- [交易日历] 开始 ---
14:35:41 | INFO    | 交易日历已覆盖到 20261231，无需更新
14:35:41 | INFO    | --- [交易日历] 完成: 0 ---
14:35:41 | INFO    | --- [股票列表] 开始 ---
14:35:41 | INFO    | 开始采集 A 股股票列表...


  运行时间: 2026-05-06 14:35:41
  模式: afternoon



14:35:41 | INFO    | 获取到 5512 只 A 股
14:35:42 | INFO    | 写入 stock_info: 5512 条
14:35:42 | INFO    | 开始采集 ETF 列表...
14:35:43 | INFO    | 获取到 1941 只 ETF
14:35:43 | INFO    | 写入 etf_info: 1941 条
14:35:43 | INFO    | --- [股票列表] 完成: stocks=5512, etfs=1941 ---
14:35:43 | INFO    | --- [个股日线] 开始 ---
14:35:47 | INFO    | 日线增量完成: 10 只, 新增 0 条
14:35:47 | INFO    | --- [个股日线] 完成: 10 只, 新增 0 条 ---
14:35:47 | INFO    | --- [ETF日线] 开始 ---
14:35:47 | INFO    | --- [ETF日线] 完成: 1 只, 新增 0 条 ---
14:35:47 | INFO    | --- [指数日线] 开始 ---
14:35:49 | INFO    | --- [指数日线] 完成: 5 指数, 新增 0 条 ---
14:35:49 | INFO    | --- [每日估值] 开始 ---
14:35:49 | INFO    | 每日估值缺失 1 天，开始增量采集
14:35:49 | INFO    | 每日估值采集完成: 1 天, 共 0 条
14:35:49 | INFO    | --- [每日估值] 完成: 0 ---
14:35:49 | INFO    | --- [技术指标] 开始 ---
14:35:50 | INFO    | 技术指标增量完成: 10 只, 新增 0 条
14:35:50 | INFO    | --- [技术指标] 完成: 10 只, 新增 0 条 ---
14:35:50 | INFO    | --- [融资融券] 开始 ---
14:35:50 | INFO    | 融资融券缺失 1 天
14:35:50 | INFO    | 融资融券采集完成: 1 天, 共 0 条
14:35:50 | INFO


运行结果汇总:
  ✅ calendar: OK (0)
  ✅ stock_list: OK (stocks=5512, etfs=1941)
  ✅ stock_daily: OK (10 只, 新增 0 条)
  ✅ etf_daily: OK (1 只, 新增 0 条)
  ✅ index_daily: OK (5 指数, 新增 0 条)
  ✅ daily_basic: OK (0)
  ✅ technical: OK (10 只, 新增 0 条)
  ✅ margin: OK (0)
  ✅ cashflow: OK (5 只, 新增 0 条)
  ✅ northbound: OK (0)
  ✅ holder_trade: OK (6)
  ✅ holder_count: OK (10)
  ✅ signal_generation: OK (date=20260430, codes=5, signals=113, composite=5, advisor=5)
  ✅ validation: OK (checks=20, passed=15, warnings=5)


### 1.3 全量运行 (等同于午盘，适用于补数据)

全部步骤一次性执行，适合首次运行或需要补全数据时使用。

In [9]:
# 全量运行 (包含所有步骤)
results_full = _run_and_print("full")

14:35:59 | INFO    | ============================================================
14:35:59 | INFO    | 每日管线启动 [full]: 2026-05-06 14:35:59
14:35:59 | INFO    | 执行步骤: ['calendar', 'stock_list', 'stock_daily', 'etf_daily', 'index_daily', 'daily_basic', 'technical', 'margin', 'cashflow', 'northbound', 'holder_trade', 'holder_count', 'signal_generation', 'validation']
14:35:59 | INFO    | ============================================================
14:35:59 | INFO    | --- [交易日历] 开始 ---
14:35:59 | INFO    | 交易日历已覆盖到 20261231，无需更新
14:35:59 | INFO    | --- [交易日历] 完成: 0 ---
14:35:59 | INFO    | --- [股票列表] 开始 ---
14:35:59 | INFO    | 开始采集 A 股股票列表...


  运行时间: 2026-05-06 14:35:59
  模式: full



14:36:00 | INFO    | 获取到 5512 只 A 股
14:36:01 | INFO    | 写入 stock_info: 5512 条
14:36:01 | INFO    | 开始采集 ETF 列表...
14:36:01 | INFO    | 获取到 1941 只 ETF
14:36:02 | INFO    | 写入 etf_info: 1941 条
14:36:02 | INFO    | --- [股票列表] 完成: stocks=5512, etfs=1941 ---
14:36:02 | INFO    | --- [个股日线] 开始 ---
14:36:04 | INFO    | 日线增量完成: 10 只, 新增 0 条
14:36:04 | INFO    | --- [个股日线] 完成: 10 只, 新增 0 条 ---
14:36:04 | INFO    | --- [ETF日线] 开始 ---
14:36:04 | INFO    | --- [ETF日线] 完成: 1 只, 新增 0 条 ---
14:36:04 | INFO    | --- [指数日线] 开始 ---
14:36:06 | INFO    | --- [指数日线] 完成: 5 指数, 新增 0 条 ---
14:36:06 | INFO    | --- [每日估值] 开始 ---
14:36:06 | INFO    | 每日估值缺失 1 天，开始增量采集
14:36:06 | INFO    | 每日估值采集完成: 1 天, 共 0 条
14:36:06 | INFO    | --- [每日估值] 完成: 0 ---
14:36:06 | INFO    | --- [技术指标] 开始 ---
14:36:07 | INFO    | 技术指标增量完成: 10 只, 新增 0 条
14:36:07 | INFO    | --- [技术指标] 完成: 10 只, 新增 0 条 ---
14:36:07 | INFO    | --- [融资融券] 开始 ---
14:36:07 | INFO    | 融资融券缺失 1 天
14:36:07 | INFO    | 融资融券采集完成: 1 天, 共 0 条
14:36:07 | INFO


运行结果汇总:
  ✅ calendar: OK (0)
  ✅ stock_list: OK (stocks=5512, etfs=1941)
  ✅ stock_daily: OK (10 只, 新增 0 条)
  ✅ etf_daily: OK (1 只, 新增 0 条)
  ✅ index_daily: OK (5 指数, 新增 0 条)
  ✅ daily_basic: OK (0)
  ✅ technical: OK (10 只, 新增 0 条)
  ✅ margin: OK (0)
  ✅ cashflow: OK (5 只, 新增 0 条)
  ✅ northbound: OK (0)
  ✅ holder_trade: OK (6)
  ✅ holder_count: OK (10)
  ✅ signal_generation: OK (date=20260430, codes=5, signals=113, composite=5, advisor=5)
  ✅ validation: OK (checks=20, passed=15, warnings=5)


## 2. 今日操作建议

管线运行完毕后，查看最新生成的操作建议。

In [10]:
import pandas as pd
from invest_model.db import get_engine
from invest_model.repositories.stock_pool_repo import StockPoolRepository
from invest_model.repositories.stock_daily_repo import StockDailyRepository
from invest_model.repositories.etf_repo import ETFRepository
from invest_model.advisor import StockAdvisor
from invest_model.advisor.persistence import load_calibration_profiles, get_advisor_history

engine = get_engine()
pool_repo = StockPoolRepository(engine)
daily_repo = StockDailyRepository(engine)
etf_repo = ETFRepository(engine)

_core_df = pool_repo.get_pool('core')
_etf_df = pool_repo.get_pool('etf')
_all_pool = pd.concat([_core_df, _etf_df], ignore_index=True)
all_codes = _all_pool['code'].tolist()
code_name_map = dict(zip(_all_pool['code'], _all_pool['name']))

latest_dates = []
for c in all_codes:
    d = daily_repo.get_latest_date(code=c)
    if not d:
        d = etf_repo.get_latest_date(c)
    if d:
        latest_dates.append(d)
trade_date = max(latest_dates) if latest_dates else datetime.now().strftime('%Y%m%d')
trade_date_fmt = f'{trade_date[:4]}-{trade_date[4:6]}-{trade_date[6:]}'

profiles = load_calibration_profiles(engine)
advisor = StockAdvisor(engine, profiles=profiles)
signals = advisor.advise_batch(all_codes, trade_date, code_name_map)

print(f'{"=" * 70}')
print(f'  📅 信号日期: {trade_date_fmt}')
print(f'  📊 标的池: {len(all_codes)} 只 (core {len(_core_df)} + etf {len(_etf_df)})')
print(f'{"=" * 70}')
print()

action_cn_map = {'strong_buy': '强买', 'buy': '买入', 'hold': '观望', 'reduce': '减仓', 'clear': '清仓'}
has_action = False

for s in signals:
    if s.action == 'hold':
        continue
    has_action = True
    icon = '🟢' if s.action in ('strong_buy', 'buy') else '🔴'
    print(f'  {icon} {s.name}({s.code}) | {action_cn_map.get(s.action, s.action)} | '
          f'置信度 {s.confidence} | 综合分 {s.composite:+.3f}')

if not has_action:
    print('  ⚪ 今日无操作建议，全部观望')

print()
print(f'{"─" * 70}')
print(f'  全部信号明细:')
print(f'{"─" * 70}')
print(f'  {"名称":8s} {"代码":12s} {"操作":4s} {"置信度":>4s} {"综合分":>7s} {"仓位":>5s}')
for s in signals:
    act = action_cn_map.get(s.action, s.action)
    print(f'  {s.name:8s} {s.code:12s} {act:4s} {s.confidence:4d} {s.composite:+7.3f} {s.position_pct:5.0%}')

  📅 信号日期: 2026-04-30
  📊 标的池: 6 只 (core 5 + etf 1)

  🔴 粤桂股份(000833.SZ) | 清仓 | 置信度 95 | 综合分 +0.349
  🟢 卫星化学(002648.SZ) | 强买 | 置信度 90 | 综合分 +0.221
  🟢 化工50ETF(516120.SH) | 强买 | 置信度 88 | 综合分 +0.382

──────────────────────────────────────────────────────────────────────
  全部信号明细:
──────────────────────────────────────────────────────────────────────
  名称       代码           操作    置信度     综合分    仓位
  粤桂股份     000833.SZ    清仓     95  +0.349   22%
  卫星化学     002648.SZ    强买     90  +0.221   17%
  化工50ETF  516120.SH    强买     88  +0.382   18%
  潞化科技     600691.SH    观望     50  +0.218    0%
  比亚迪      002594.SZ    观望     31  +0.024    0%
  润泽科技     300442.SZ    观望     31  -0.019    0%


In [11]:
# 详细归因报告
report_df = advisor.daily_report(all_codes, trade_date, code_name_map)
if not report_df.empty:
    print(f'📋 逐票归因 ({trade_date_fmt})\n')
    for _, row in report_df.iterrows():
        act = row['操作']
        if act == '观望':
            continue
        print(f'{"─" * 70}')
        print(f'  [{row["名称"]}({row["代码"]})] {act} (置信度{row["置信度"]})')
        print(f'  {row["归因"]}')
    
    non_hold = report_df[report_df['操作'] != '观望']
    if non_hold.empty:
        print('  今日全部观望，无需操作。')
    print(f'{"─" * 70}')
else:
    print('无数据')

📋 逐票归因 (2026-04-30)

──────────────────────────────────────────────────────────────────────
  [粤桂股份(000833.SZ)] 清仓 (置信度95)
  整体偏多（综合 +0.35）。看多：MACD 多头强势，趋势向上、5日涨幅强劲(27.03%)，短线动量充足；看空：MA60 大幅偏离上方 (29.56%)，均值回归概率增大、价格接近布林上轨 (100%)，波动加大。 | 置信度95分: 趋势=bullish,阶段=acceleration,z=+11.74; 趋势奖励+15; 一致性+20 | 触发: 价格偏离MA60 +29.6%，超过+15%阈值，注意止盈; 价格处于近20日最高区间(距高点0%)，注意止盈 | 过滤: 止盈覆盖:MA60偏离过大+20日高点; comp持续回升(Δ=+0.122,-15)
──────────────────────────────────────────────────────────────────────
  [卫星化学(002648.SZ)] 强买 (置信度90)
  整体偏多（综合 +0.22）。看多：PE 处于行业低位 (分位 2%，PE=17.0)、MACD 多头强势，趋势向上；看空：价格接近布林上轨 (100%)，波动加大、股东户数较上期下降 15.12%，筹码明显集中。 | 置信度90分: 趋势=bullish,阶段=acceleration,z=+5.77; 趋势奖励+15; 一致性+15 | 过滤: comp加速改善(Δ=+0.101,+12)
──────────────────────────────────────────────────────────────────────
  [化工50ETF(516120.SH)] 强买 (置信度88)
  整体偏多（综合 +0.38）。看多：北向资金 5 日累计净流入 169.1 亿，外资大幅加仓、5日小幅上涨(6.26%)，短线偏多；看空：MACD 死叉，需关注下行风险。 | 置信度88分: 趋势=bullish_weak,阶段=acceleration,comp=+0.382; 趋势奖励+8; 一致性+20 | 触发: 价格处于近20日最高区间(距高点3%

## 3. cron 定时任务配置

### 命令行运行
```bash
cd /home/admin/Code/invest-journey/invest-model

# 完整管线
python3 -m invest_model.pipeline.daily_pipeline full

# 只跑早盘
python3 -m invest_model.pipeline.daily_pipeline morning

# 只跑午盘
python3 -m invest_model.pipeline.daily_pipeline afternoon
```

### crontab 配置
周一到周五，早盘收盘后 11:35 和午盘收盘后 15:05 各执行一次：

```bash
# 编辑 crontab
crontab -e

# 添加以下两行
35 11 * * 1-5 cd /home/admin/Code/invest-journey/invest-model && /usr/bin/python3 -m invest_model.pipeline.daily_pipeline morning   >> logs/cron.log 2>&1
 5 15 * * 1-5 cd /home/admin/Code/invest-journey/invest-model && /usr/bin/python3 -m invest_model.pipeline.daily_pipeline afternoon >> logs/cron.log 2>&1
```

### 验证 crontab
```bash
crontab -l
```

## 4. 查看运行日志

In [12]:
from pathlib import Path
from invest_model.config import get_project_root

log_dir = get_project_root() / "logs"
log_files = sorted(log_dir.glob("invest-model-*.log"), reverse=True)

if log_files:
    latest = log_files[0]
    print(f"最新日志: {latest.name}")
    lines = latest.read_text().strip().split("\n")
    print(f"共 {len(lines)} 行，最后 20 行:\n")
    for line in lines[-20:]:
        print(line)
else:
    print("暂无日志文件")

最新日志: invest-model-20260506.log
共 1334 行，最后 20 行:

2026-05-06 14:36:15 | INFO    | daily_pipeline:_run_step:137 | --- [综合评分] 完成: date=20260430, codes=5, signals=113, composite=5, advisor=5 ---
2026-05-06 14:36:15 | INFO    | daily_pipeline:_run_step:135 | --- [数据校验] 开始 ---
2026-05-06 14:36:16 | INFO    | daily_pipeline:_run_step:137 | --- [数据校验] 完成: checks=20, passed=15, warnings=5 ---
2026-05-06 14:36:16 | INFO    | daily_pipeline:run:124 | ============================================================
2026-05-06 14:36:16 | INFO    | daily_pipeline:run:125 | 每日管线完成 [full]，耗时 16.4s
2026-05-06 14:36:16 | INFO    | daily_pipeline:run:127 |   calendar: OK (0)
2026-05-06 14:36:16 | INFO    | daily_pipeline:run:127 |   stock_list: OK (stocks=5512, etfs=1941)
2026-05-06 14:36:16 | INFO    | daily_pipeline:run:127 |   stock_daily: OK (10 只, 新增 0 条)
2026-05-06 14:36:16 | INFO    | daily_pipeline:run:127 |   etf_daily: OK (1 只, 新增 0 条)
2026-05-06 14:36:16 | INFO    | daily_pipeline:run:127 |   in

## 完成

每日管线已配置完毕。数据层全部 10 个 Notebook 已完成：

| 序号 | Notebook | 状态 |
|------|---------|------|
| 00 | 环境配置 | Done |
| 01 | 数据库初始化 | Done |
| 02 | 股票池管理 | Done |
| 03 | 日线行情 | Done |
| 04 | 财务指标 | Done |
| 05 | ETF 数据 | Done |
| 06 | 事件数据 | Done |
| 07 | 市场数据 | Done |
| 08 | 数据校验 | Done |
| 09 | 每日管线 | Done |